# Zéro, une ou une infinité de solutions

| Module | Durée | Prérequis |
|---|---|---|
| M2 – Systèmes linéaires | 75 min | M1 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

**Les forfaits.** Léa compare des forfaits téléphoniques sans engagement :

| Forfait | Abonnement mensuel | Prix par minute |
|---|---|---|
| A | 10 € | 0,05 € |
| B | 4 € | 0,08 € |
| C | 6 € | 0,05 € |
| D | 20 € les deux mois | 0,10 € les deux minutes |

Elle se demande : **pour quelle durée d'appel deux forfaits coûtent-ils exactement le même prix ?**

**Le repas.** Un sportif veut composer un repas qui apporte exactement 25 g de protéines, 80 g de glucides et 15 g de lipides. Il dispose de lentilles, de riz et d'amandes. **Quelles quantités prendre ?**

Dans les deux cas, on va voir qu'un système peut avoir **une** solution, **aucune**, ou **une infinité**, et surtout ce que chaque cas signifie concrètement.

**Activité** : Avant tout calcul : pour chaque paire de forfaits (A et B, A et C, A et D), pensez-vous qu'il existe une durée où les prix sont égaux ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
# Pour chaque paire, pariez : "aucune", "une" ou "une infinité"
pari_A_B = "une"
pari_A_C = "une"
pari_A_D = "une"

## ③ Abstraire

Pour un forfait, le prix $p$ (en €) dépend de la durée $m$ (en minutes) :

$$
p = \text{abonnement} + \text{prix par minute} \times m
$$

C'est l'équation d'une **droite**. Chercher une durée où deux forfaits coûtent autant, c'est chercher un point commun à deux droites. On écrit le système sous la forme « inconnues $m$ et $p$ à gauche » :

$$
\begin{cases}
-0{,}05\,m + p = 10 & \text{(forfait A)} \\
-0{,}08\,m + p = 4 & \text{(forfait B)}
\end{cases}
$$

**Géométriquement**, deux droites du plan peuvent :

- se couper en un point : **une** solution ;
- être parallèles et distinctes : **aucune** solution ;
- être confondues : **une infinité** de solutions.

![Schéma : deux droites sécantes (une solution), parallèles (aucune), confondues (une infinité)](../../assets/schemas/schema-trois-cas.svg)

**Activité** : Écrivez l'équation du forfait D par minute. Que remarquez-vous ?

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Les forfaits, en image

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

forfaits = {"A": (10, 0.05), "B": (4, 0.08), "C": (6, 0.05), "D": (20 / 2, 0.10 / 2)}   # D ramené au mois et à la minute
minutes = np.linspace(0, 400, 200)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, (f1, f2), titre in zip(axes, [("A", "B"), ("A", "C"), ("A", "D")], ["A et B", "A et C", "A et D"]):
    for nom, style in [(f1, "-"), (f2, "--")]:
        abo, prix_min = forfaits[nom]
        ax.plot(minutes, abo + prix_min * minutes, style, lw=2.5 if style == "--" else 4, label=f"forfait {nom}",
                alpha=0.9 if style == "--" else 0.5)
    ax.set_title(titre); ax.set_xlabel("minutes par mois"); ax.legend(); ax.grid(alpha=0.3)
axes[0].set_ylabel("prix mensuel (€)")
plt.show()

### Un diagnostic par le calcul : le rang

Un dessin suffit en dimension 2, plus en dimension 10. Il faut un critère calculable. Le **rang** d'une matrice est le nombre de pivots obtenus par le pivot de Gauss, autrement dit le nombre d'équations vraiment indépendantes.

On compare trois nombres : le rang de $A$, noté $\operatorname{rg} A$, le rang de la matrice augmentée $[A \mid \mathbf{b}]$ et le nombre d'inconnues $n$.

| Situation | Signification | Nombre de solutions |
|---|---|---|
| $\operatorname{rg}[A \mid \mathbf{b}] > \operatorname{rg} A$ | une équation contredit les autres (on obtient $0 = 5$) | aucune |
| $\operatorname{rg}[A \mid \mathbf{b}] = \operatorname{rg} A = n$ | assez d'équations indépendantes, aucune contradiction | une |
| $\operatorname{rg}[A \mid \mathbf{b}] = \operatorname{rg} A < n$ | pas de contradiction, mais trop peu d'équations | une infinité |

In [ ]:
def diagnostic(A, b):
    A = np.atleast_2d(np.array(A, dtype=float))
    b = np.array(b, dtype=float)
    r_A = np.linalg.matrix_rank(A)
    r_Ab = np.linalg.matrix_rank(np.column_stack([A, b]))
    n = A.shape[1]
    if r_Ab > r_A:
        verdict = "aucune solution"
    elif r_A == n:
        verdict = "une seule solution"
    else:
        verdict = f"une infinité de solutions ({n - r_A} inconnue(s) libre(s))"
    return r_A, r_Ab, n, verdict

def systeme_forfaits(f1, f2):
    (abo1, pm1), (abo2, pm2) = forfaits[f1], forfaits[f2]
    return [[-pm1, 1], [-pm2, 1]], [abo1, abo2]

for f1, f2 in [("A", "B"), ("A", "C"), ("A", "D")]:
    A, b = systeme_forfaits(f1, f2)
    r_A, r_Ab, n, verdict = diagnostic(A, b)
    print(f"{f1} et {f2} : rang A = {r_A}, rang [A|b] = {r_Ab}, n = {n}  ->  {verdict}")
    if verdict == "une seule solution":
        m, p = np.linalg.solve(A, b)
        print(f"         même prix pour {m:.0f} minutes : {p:.2f} €")

### Le repas : trois inconnues

Valeurs nutritionnelles arrondies, pour 100 g d'aliment prêt à manger :

| Aliment | Protéines (g) | Glucides (g) | Lipides (g) |
|---|---|---|---|
| lentilles cuites | 9 | 20 | 0,4 |
| riz cuit | 2,7 | 28 | 0,3 |
| amandes | 21 | 22 | 50 |

Les inconnues sont les quantités, en centaines de grammes. Chaque nutriment donne une équation : géométriquement, un **plan** de l'espace. Trois plans peuvent avoir un point commun, une droite commune, ou rien du tout.

In [ ]:
# Colonnes : lentilles, riz, amandes ; lignes : protéines, glucides, lipides
N = np.array([
    [9,   2.7, 21],
    [20,  28,  22],
    [0.4, 0.3, 50],
])
cible = np.array([25, 80, 15])

print(diagnostic(N, cible)[3])
q = np.linalg.solve(N, cible)
for aliment, quantite in zip(["lentilles", "riz", "amandes"], q):
    print(f"{aliment:10s} : {100 * quantite:5.0f} g")

**Un placard différent.** Le sportif n'a plus d'amandes, mais un sachet de « salade lentilles-riz » toute prête : moitié lentilles, moitié riz. Son troisième vecteur est donc une combinaison linéaire des deux premiers.

In [ ]:
N2 = N.copy()
N2[:, 2] = 0.5 * N[:, 0] + 0.5 * N[:, 1]          # salade = 50 % lentilles + 50 % riz

print("Même cible     :", diagnostic(N2, cible)[3])

cible_atteignable = N2 @ np.array([1, 1, 1])        # un repas qu'on sait composer : 100 g de chaque
print("Cible", cible_atteignable, ":", diagnostic(N2, cible_atteignable)[3])

Dans le cas « infinité », toutes les solutions s'obtiennent à partir d'une seule, $\mathbf{x}_0$, en lui ajoutant n'importe quel multiple d'un vecteur $\mathbf{n}$ du **noyau** : une combinaison d'aliments qui n'apporte rien au total.

$$
\mathbf{x} = \mathbf{x}_0 + s\,\mathbf{n}, \qquad N\mathbf{n} = \mathbf{0}, \qquad s \in \mathbb{R}
$$

In [ ]:
from scipy.linalg import null_space

x0 = np.array([1, 1, 1])
direction = null_space(N2)[:, 0]
direction = direction / direction[2]               # normalisée pour que la salade varie de 1
print("Direction du noyau :", direction.round(3))

for s in [-1, 0, 1]:
    x = x0 + s * direction
    print(f"s = {s:+d} : lentilles {100 * x[0]:5.0f} g, riz {100 * x[1]:5.0f} g, salade {100 * x[2]:5.0f} g"
          f"  ->  apports {np.round(N2 @ x, 2)}")

## ⑤ Interpréter

In [ ]:
for paire, pari in [("A et B", pari_A_B), ("A et C", pari_A_C), ("A et D", pari_A_D)]:
    print(f"{paire} : votre pari « {pari} »")

**Activité** : Traduisez chaque résultat dans le langage de la personne concernée.

1. **A et B** : que conseiller à Léa selon qu'elle téléphone plus ou moins de 200 minutes par mois ?
2. **A et C** : que signifie « aucune solution » pour Léa ? Lequel des deux forfaits est toujours meilleur ?
3. **A et D** : que signifie « une infinité de solutions » ? Le forfait D est-il vraiment différent du forfait A ?
4. **Le repas avec amandes** : la solution est-elle réaliste ? Toutes les quantités sont-elles positives ?
5. **Le repas avec salade et la cible d'origine** : pourquoi est-ce impossible ? Quel nutriment pose problème ?
6. **Le repas avec salade et la cible atteignable** : les trois lignes affichées apportent exactement les mêmes nutriments. Pour quelles valeurs de $s$ une quantité devient-elle négative ? Parmi les repas possibles, comment choisir ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Hypothèse fragile.** Les valeurs nutritionnelles sont arrondies et varient selon les marques. Avec le système « salade », une petite erreur sur la cible fait-elle passer de « infinité » à « aucune solution » ? Testez en changeant la cible de 0,01 g. Que vous apprend ce test sur l'usage du rang avec des données mesurées ?
- **Vérifier autrement.** Pour A et B, vérifiez la durée trouvée en calculant le prix de chaque forfait à cette durée.
- **Généraliser.** Une « infinité » appelle un **critère** pour choisir : le moins cher, le plus léger, le plus équilibré. Une « aucune » appelle un **compromis** : la solution la plus proche de la cible. Ces deux idées reviendront, la seconde au module M6.

In [ ]:
# Votre test de sensibilité

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
cible_bruitee = cible_atteignable + np.array([0, 0, 0.01])
print("Cible atteignable + 0,01 g de lipides :", diagnostic(N2, cible_bruitee)[3])
print("Rang de [A|b] avec une tolérance de 0,1 au lieu de la tolérance par défaut :",
      np.linalg.matrix_rank(np.column_stack([N2, cible_bruitee]), tol=0.1))
print("Le rang est un critère « tout ou rien » : avec des données mesurées, il faut fixer une tolérance,")
print("c'est-à-dire décider à partir de quel écart une contradiction est réelle.")

m, p = np.linalg.solve(*systeme_forfaits("A", "B"))
print(f"\nÀ {m:.0f} min : A coûte {10 + 0.05 * m:.2f} €, B coûte {4 + 0.08 * m:.2f} €")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Énoncer le résultat dans le contexte du problème": 1,
    "Relier les traits du résultat à leur sens réel": 1,
    "Reconnaître si le concept s'applique": 1,
    "Quantifier la validité et l'impact des hypothèses": 1,
    "Estimer la solution avant de calculer": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "2.3-nombre-de-solutions", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")